In [1]:
#Task 4

import sqlite3

#Task 4.1

conn = sqlite3.connect('VillaBookings.db')

conn.execute('''
CREATE TABLE IF NOT EXISTS "Villa" (
	"villa_ID"	INTEGER NOT NULL,
	"villa_name"	TEXT NOT NULL,
	"country"	TEXT NOT NULL,
	"cost"	INTEGER,
	PRIMARY KEY("villa_ID")
);''')

conn.execute('''
CREATE TABLE IF NOT EXISTS"Booking" (
	"booking_ID"	INTEGER NOT NULL,
	"customer_ID"	INTEGER NOT NULL,
	"villa_ID"	INTEGER NOT NULL,
	"start_date"	TEXT NOT NULL,
	"number_of_days"	INTEGER NOT NULL,
	PRIMARY KEY("booking_ID","customer_ID"),
	FOREIGN KEY("villa_ID") REFERENCES "Villa"("villa_ID")
);''')

conn.commit()
conn.close()



In [2]:

#Task 4.2
def file():
    conn = sqlite3.connect('VillaBookings.db')

    file = open('villas.txt','r')
    for line in file:
        line = line.strip()
        line = line.split(',')
        villa_ID,villa_name,country,cost = line[0],line[1],line[2],line[3]
        conn.execute('''INSERT INTO Villa(villa_ID,villa_name,country,cost)
        VALUES (?,?,?,?)''',(villa_ID,villa_name,country,cost))
    
    conn.commit()
    file.close()

    conn = sqlite3.connect('VillaBookings.db')
    file = open('customerBookings.txt','r')
    for line in file:
        line = line.strip()
        line = line.split(',')
        booking_ID,customer_ID,villa_ID,start_date,number_of_days = line[0],line[1],line[2],line[3],line[4]
        conn.execute(''' INSERT INTO Booking(booking_ID,customer_ID,villa_ID,start_date,number_of_days)
        VALUES(?,?,?,?,?)''',(booking_ID,customer_ID,villa_ID,start_date,number_of_days))

    conn.commit()
    file.close()



In [3]:
#Task 4.3

conn = sqlite3.connect('VillaBookings.db')
conn.execute('''CREATE TABLE IF NOT EXISTS "Villa_Booking" (
	"villa_ID"	INTEGER NOT NULL,
	"date"	TEXT NOT NULL,
	FOREIGN KEY("villa_ID") REFERENCES "Villa"("villa_ID")
);''')
conn.commit()

result = conn.execute('''SELECT Villa.villa_ID,Booking.number_of_days,Booking.start_date FROM Villa,Booking
    WHERE Villa.villa_ID = Booking.villa_ID''').fetchall()
print(result)

for line in result:
    villa_ID,number_of_days,start_date = line[0],line[1],line[2]
    month_lst = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    for i in range(number_of_days):
        date = line[2].split('-')[0]
        month = line[2].split('-')[1]
        date = int(date)
        date = date + int(i)
        if date > 31:
            for j in range(12):
                if month == month_lst[j]:
                    month = month_lst[j+1]
                    break
            date = date - 31
        date_string = str(date) + "-" + month

        conn.execute('''INSERT INTO Villa_Booking(villa_ID,date)
        VALUES(?,?)''',(villa_ID,date_string))

conn.commit()



[(3, 7, '05-Jan'), (10, 11, '20-Jan'), (9, 2, '05-Feb'), (4, 14, '03-Mar'), (6, 10, '19-Nov'), (4, 4, '12-May'), (7, 6, '03-Mar'), (2, 2, '18-Aug'), (3, 4, '02-Mar'), (5, 7, '04-Apr'), (10, 12, '07-Jul'), (8, 5, '15-Jan'), (9, 4, '06-Nov'), (9, 7, '03-Apr'), (5, 14, '12-Oct'), (4, 6, '08-Jul'), (4, 7, '09-Aug'), (1, 3, '12-Nov'), (1, 4, '15-Sep'), (2, 14, '03-Feb'), (3, 14, '10-Apr'), (3, 2, '01-Feb'), (6, 21, '13-Apr'), (4, 3, '02-Nov'), (5, 4, '21-Dec'), (7, 9, '12-Aug'), (8, 10, '06-Jun'), (2, 4, '06-May'), (2, 3, '09-Oct'), (3, 7, '28-Sep'), (8, 5, '12-Sep'), (3, 3, '16-Jan'), (1, 4, '03-Jan'), (1, 8, '03-Dec'), (1, 2, '06-Jun')]


In [5]:
#Task 4.4

def check_availability(villa_name,start_date,number_of_days):
    conn = sqlite3.connect('VillaBookings.db')
    date_lst = []
    available_dates = []
    unavailable_dates = []
    
    month_lst = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    result = conn.execute('''SELECT Villa_Booking.date FROM Villa_Booking,Villa
    WHERE Villa_Booking.villa_ID = Villa.villa_ID
    AND Villa.villa_name = ?
    ;''',(villa_name,)).fetchall()
    
    for i in range(number_of_days):
        count = 0
        date = start_date.split('-')[0]
        month = start_date.split('-')[1]
        date = int(date)
        date = date + int(i)
        if date > 31:
            for j in range(12):
                if month == month_lst[j]:
                    month = month_lst[j+1]
                    break
            date = date - 31
        date_string = str(date) + "-" + month
        print(date_string)
    
        for j in range(len(result)):
            if result[j][0] == date_string:
                unavailable_dates.append(result[j][0])
                count += 1
                break
                
        if count == 0:
            available_dates.append(date_string)
    print("Available on these days",available_dates)
    print("Unavailable on these days",unavailable_dates)
    return 
    

check_availability('Dolphin','08-Apr',4)


    
        
    

    


8-Apr
9-Apr
10-Apr
11-Apr
Available on these days ['8-Apr', '9-Apr']
Unavailable on these days ['10-Apr', '11-Apr']
